# Notebook de Veille Technique — XAI (Explicabilité des Modèles)
## Projet 8 OpenClassrooms — Dominique
### Comparaison SHAP vs LIME sur le modèle de scoring crédit

---

## Objectif

Ce notebook compare deux méthodes d'explicabilité (XAI) :
- **SHAP** (SHapley Additive exPlanations) — Lundberg & Lee, 2017
- **LIME** (Local Interpretable Model-agnostic Explanations) — Ribeiro et al., 2016

Appliquées au modèle LightGBM de scoring crédit (Projet 7).

**Références :**
- SHAP : https://arxiv.org/abs/1705.07874
- LIME : https://arxiv.org/abs/1602.04938

## 1. Imports et configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
import time
import json

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams['figure.facecolor'] = 'white'

print('✅ Imports de base effectués')

In [ ]:
# Vérification des librairies XAI
try:
    import shap
    print(f'✅ SHAP version : {shap.__version__}')
except ImportError:
    print('❌ SHAP non installé → pip install shap')

try:
    import lime
    import lime.lime_tabular
    print(f'✅ LIME version : {lime.__version__}')
except ImportError:
    print('⚠️ LIME non installé → pip install lime')
    print('   (Optionnel pour ce notebook)')

try:
    import lightgbm as lgb
    print(f'✅ LightGBM version : {lgb.__version__}')
except ImportError:
    print('❌ LightGBM non installé')

## 2. Chargement du modèle entraîné (Projet 7)

In [ ]:
import pickle
from pathlib import Path

# Chemins vers les artefacts du Projet 7
MODEL_PATH = Path(r'C:\Users\domir\Documents\projet7_Open_Dominique\models\model.pkl')
PREP_PATH = Path(r'C:\Users\domir\Documents\projet7_Open_Dominique\models\preprocessor.pkl')
CONFIG_PATH = Path(r'C:\Users\domir\Documents\projet7_Open_Dominique\models\model_config.json')

# Chargement
with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)

with open(PREP_PATH, 'rb') as f:
    preprocessor = pickle.load(f)

with open(CONFIG_PATH, 'r') as f:
    config = json.load(f)

print(f'✅ Modèle chargé : {config["model_type"]}')
print(f'   AUC-ROC      : {config.get("auc_roc", "N/A")}')
print(f'   Seuil optimal: {config["threshold"]}')
print(f'   Nb features  : {len(config["feature_names"])}')

## 3. Création d'un jeu de données de test simulé

In [ ]:
np.random.seed(42)
n = 100

# Profils clients simulés
data = {
    'AMT_INCOME_TOTAL': np.random.lognormal(11.5, 0.5, n),
    'AMT_CREDIT': np.random.lognormal(12.5, 0.6, n),
    'AMT_ANNUITY': np.random.lognormal(9.5, 0.4, n),
    'AMT_GOODS_PRICE': np.random.lognormal(12.3, 0.6, n),
    'DAYS_BIRTH': np.random.randint(-25000, -6000, n),
    'DAYS_EMPLOYED': np.random.randint(-10000, -100, n),
    'CNT_CHILDREN': np.random.randint(0, 4, n),
    'CODE_GENDER_M': np.random.randint(0, 2, n),
    'FLAG_OWN_CAR': np.random.randint(0, 2, n),
    'FLAG_OWN_REALTY': np.random.randint(0, 2, n),
    'EXT_SOURCE_1': np.random.beta(3, 2, n),
    'EXT_SOURCE_2': np.random.beta(4, 2, n),
    'EXT_SOURCE_3': np.random.beta(3, 2, n),
    'REGION_RATING_CLIENT': np.random.randint(1, 4, n),
}

df = pd.DataFrame(data)

# Ajout des features dérivées
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1)
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1)
df['EXT_SOURCE_MEAN'] = df[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].mean(axis=1)

print(f'✅ Dataset simulé : {df.shape}')
df.head(3)

## 4. Prédictions du modèle

In [ ]:
import sys
sys.path.insert(0, r'C:\Users\domir\Documents\projet7_Open_Dominique')
from src.preprocessing import create_application_features

# Feature engineering
df_processed = create_application_features(df.copy())

# Aligner avec les features du modèle
feature_names = config['feature_names']
for col in feature_names:
    if col not in df_processed.columns:
        df_processed[col] = np.nan

X = df_processed[feature_names]
X_proc = preprocessor.transform(X)

# Prédictions
y_proba = model.predict_proba(X_proc)[:, 1]
threshold = config['threshold']
y_pred = (y_proba >= threshold).astype(int)

print(f'✅ Prédictions effectuées')
print(f'   Taux de refus  : {y_pred.mean():.1%}')
print(f'   Proba moyenne  : {y_proba.mean():.3f}')
print(f'   Proba min/max  : {y_proba.min():.3f} / {y_proba.max():.3f}')

## 5. Méthode 1 — SHAP (SHapley Additive exPlanations)

In [ ]:
import shap

# Créer l'explainer SHAP (TreeExplainer optimisé pour LightGBM)
start = time.time()
explainer_shap = shap.TreeExplainer(model)
shap_values = explainer_shap.shap_values(X_proc)
duration_shap = time.time() - start

# Pour LightGBM binaire, shap_values est une liste [classe_0, classe_1]
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

print(f'✅ SHAP calculé en {duration_shap:.2f}s pour {n} clients')
print(f'   Shape SHAP values : {sv.shape}')

In [ ]:
# Summary plot SHAP (importance globale)
plt.figure(figsize=(10, 8))
shap.summary_plot(
    sv, 
    pd.DataFrame(X_proc, columns=feature_names),
    max_display=15,
    show=False
)
plt.title('SHAP — Importance globale des features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_summary_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Summary plot sauvegardé')

In [ ]:
# Explication locale SHAP pour un client spécifique
client_idx = 0
client_shap = sv[client_idx]
client_features = pd.Series(X_proc[client_idx], index=feature_names)

# Top 10 features pour ce client
shap_df = pd.DataFrame({
    'feature': feature_names,
    'shap_value': client_shap,
    'feature_value': X_proc[client_idx]
}).sort_values('shap_value', key=abs, ascending=False).head(10)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#dc3545' if v > 0 else '#28a745' for v in shap_df['shap_value']]
ax.barh(shap_df['feature'][::-1], shap_df['shap_value'][::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Valeur SHAP (impact sur la prédiction)')
ax.set_title(f'SHAP — Explication locale (Client #{client_idx}, Proba={y_proba[client_idx]:.3f})')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('shap_local_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Méthode 2 — LIME

In [ ]:
try:
    import lime
    import lime.lime_tabular

    start = time.time()
    explainer_lime = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_proc,
        feature_names=feature_names,
        class_names=['Non-défaut', 'Défaut'],
        mode='classification',
        random_state=42
    )

    # Explication pour le même client
    exp = explainer_lime.explain_instance(
        X_proc[client_idx],
        model.predict_proba,
        num_features=10,
        num_samples=500,
    )
    duration_lime = time.time() - start

    # Visualisation LIME
    lime_vals = exp.as_list(label=1)
    lime_df = pd.DataFrame(lime_vals, columns=['feature_condition', 'weight'])
    lime_df = lime_df.sort_values('weight', key=abs, ascending=False)

    fig, ax = plt.subplots(figsize=(9, 5))
    colors = ['#dc3545' if v > 0 else '#28a745' for v in lime_df['weight']]
    ax.barh(lime_df['feature_condition'][::-1], lime_df['weight'][::-1], color=colors[::-1])
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlabel('Poids LIME (impact sur la prédiction)')
    ax.set_title(f'LIME — Explication locale (Client #{client_idx})')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('lime_local_plot.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'✅ LIME calculé en {duration_lime:.2f}s')

except ImportError:
    print('⚠️ LIME non installé — installez avec : pip install lime')
    duration_lime = None

## 7. Comparaison SHAP vs LIME

In [ ]:
# Tableau de comparaison
comparaison = pd.DataFrame({
    'Critère': [
        'Fondement théorique',
        'Cohérence mathématique',
        'Stabilité (même input)',
        'Vitesse (100 clients)',
        'Adapté à LightGBM',
        'Explication globale',
        'Explication locale',
        'Adoption industrielle',
    ],
    'SHAP': [
        'Valeurs de Shapley (théorie des jeux)',
        '✅ Garantie (additivité, nullité)',
        '✅ Déterministe',
        f'✅ {duration_shap:.2f}s (TreeExplainer)',
        '✅ TreeExplainer natif',
        '✅ Summary plot',
        '✅ Force plot, waterfall',
        '✅ Très large',
    ],
    'LIME': [
        'Approximation locale linéaire',
        '⚠️ Non garantie',
        '⚠️ Variable (perturbation aléatoire)',
        f'{f"✅ {duration_lime:.2f}s" if duration_lime else "N/A (non installé)"}',
        '✅ Compatible',
        '❌ Local uniquement',
        '✅ Coefficients linéaires',
        '✅ Large',
    ]
})

print(comparaison.to_string(index=False))

## 8. Conclusion

### Méthode retenue : **SHAP**

Pour le contexte bancaire et réglementaire de Prêt à Dépenser :

1. **SHAP** offre des garanties mathématiques (valeurs de Shapley) que LIME ne peut pas fournir
2. **TreeExplainer** est optimisé nativement pour LightGBM → rapide en production
3. Les explications sont **déterministes** → important pour la conformité RGPD
4. SHAP permet des explications **globales et locales** cohérentes

### Features les plus importantes identifiées

- `EXT_SOURCE_2`, `EXT_SOURCE_3`, `EXT_SOURCE_1` → scores externes de solvabilité
- `CREDIT_INCOME_RATIO` → ratio crédit/revenu (sur-endettement)
- `DAYS_BIRTH` → âge du client

Ces résultats sont cohérents avec la théorie financière.

---
*Notebook réalisé dans le cadre du Projet 8 OpenClassrooms*  
*Dominique (Dom130) — Veille technique XAI*